<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 25 · PyTorch căn bản</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Tensor, autograd, nn.Module và vòng huấn luyện bốn dòng, DataLoader và GPU: viết lại mạng của Chương 24 bằng PyTorch rồi phân loại chữ số viết tay.</p>
</div>

### Sau notebook này bạn sẽ

- Dùng tensor như mảng NumPy và tránh bẫy dùng chung bộ nhớ.
- Tính gradient bằng autograd, kiểm bằng số của Chương 24.
- Viết vòng huấn luyện zero_grad, loss, backward, step.
- Huấn luyện theo lô bằng DataLoader tới 95,8% trên chữ số viết tay.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics.vercel.app/ai-engineer/ch25-pytorch-can-ban/01-tensor/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Tensor](#trang-1)
2. [Trang 2 · Autograd](#trang-2)
3. [Trang 3 · nn.Module và vòng huấn luyện](#trang-3)
4. [Trang 4 · DataLoader, nhiều lớp và GPU](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Tensor

Tensor như mảng NumPy.

*Đọc kèm trên web: [Trang 1 · Tensor](https://ai-and-robotics.vercel.app/ai-engineer/ch25-pytorch-can-ban/01-tensor/)*

In [ ]:
import torch                       # PyTorch
import numpy as np                 # để so sánh
print(torch.__version__)           # phiên bản
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])   # tensor 2x2
print(t.shape, t.dtype)            # shape và kiểu
print(t @ t)                       # nhân ma trận, như NumPy
print(t.T)                         # chuyển vị
print(t.sum(0))                    # tổng theo trục 0

Output mong đợi:

```text
2.14.1+cpu
torch.Size([2, 2]) torch.float32
tensor([[ 7., 10.],
        [15., 22.]])
tensor([[1., 3.],
        [2., 4.]])
tensor([4., 6.])
```

In [ ]:
a = np.array([1.0, 2.0])           # mảng NumPy
ta = torch.from_numpy(a)           # KHÔNG chép: dùng chung bộ nhớ
ta[0] = 99                         # sửa qua tensor
print(a)                           # mảng NumPy cũng đổi

Output mong đợi:

```text
[99.  2.]
```

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO MẢNG NUMPY BỊ ĐỔI?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">torch.from_numpy và .numpy() không chép dữ liệu. Muốn bản sao độc lập, dùng torch.tensor(a) hoặc .clone().</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Autograd

Autograd: số của Chương 24, không viết lượt ngược.

*Đọc kèm trên web: [Trang 2 · Autograd](https://ai-and-robotics.vercel.app/ai-engineer/ch25-pytorch-can-ban/02-autograd/)*

In [ ]:
w = torch.tensor(0.5, requires_grad=True)       # cần gradient theo w
b = torch.tensor(0.1, requires_grad=True)       # cần gradient theo b
x, y = 2.0, 1.0                                 # đầu vào, nhãn
L = (torch.sigmoid(w * x + b) - y) ** 2         # lượt xuôi
L.backward()                                    # lượt ngược tự động
print(round(L.item(), 4), round(w.grad.item(), 4), round(b.grad.item(), 4))   # loss và gradient

Output mong đợi:

```text
0.0624 -0.1872 -0.0936
```

In [ ]:
w = torch.tensor(0.5, requires_grad=True)       # tạo lại w
for lan in range(3):                            # ba lần backward
    L = (torch.sigmoid(w * 2.0 + 0.1) - 1.0) ** 2   # lượt xuôi
    L.backward()                                # không xóa gradient
    print(lan, round(w.grad.item(), 4))         # gradient cộng dồn

Output mong đợi:

```text
0 -0.1872
1 -0.3743
2 -0.5615
```

In [ ]:
with torch.no_grad():                           # tắt autograd
    z = w * 2.0 + 0.1                           # không ghi đồ thị
print(z.requires_grad)                          # False

Output mong đợi:

```text
False
```

<a name="trang-3" id="trang-3"></a>

## Trang 3 · nn.Module và vòng huấn luyện

nn.Sequential và vòng huấn luyện bốn dòng.

*Đọc kèm trên web: [Trang 3 · nn.Module và vòng huấn luyện](https://ai-and-robotics.vercel.app/ai-engineer/ch25-pytorch-can-ban/03-nn-module/)*

In [ ]:
import torch.nn as nn                           # các lớp mạng
torch.manual_seed(0)                            # khởi tạo lặp lại được
lin = nn.Linear(2, 3)                           # 2 đầu vào, 3 nơ-ron
print(lin.weight.shape, lin.bias.shape, sum(p.numel() for p in lin.parameters()))   # shape và số tham số

Output mong đợi:

```text
torch.Size([3, 2]) torch.Size([3]) 9
```

In [ ]:
from sklearn.datasets import make_moons                     # dữ liệu
from sklearn.model_selection import train_test_split        # chia
Xm, ym = make_moons(300, noise=0.3, random_state=0)         # hai vầng trăng
a, b_, c, e = train_test_split(Xm, ym, test_size=0.3, random_state=0)   # 210 train, 90 test
Xtr, ytr = torch.tensor(a, dtype=torch.float32), torch.tensor(c, dtype=torch.float32)   # train
Xte, yte = torch.tensor(b_, dtype=torch.float32), torch.tensor(e, dtype=torch.float32)  # test
torch.manual_seed(0)                                        # seed
m = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))   # 2 -> 32 ReLU -> 1
print(sum(p.numel() for p in m.parameters()))               # số tham số

Output mong đợi:

```text
129
```

In [ ]:
opt = torch.optim.SGD(m.parameters(), lr=0.5)      # gradient descent
lf = nn.BCEWithLogitsLoss()                        # sigmoid + cross-entropy
dung = lambda X, y: ((m(X).squeeze(1) > 0).float() == y).float().mean().item()   # độ chính xác
for buoc in range(3001):                           # 3000 bước
    if buoc in (0, 100, 1000, 3000):               # in vài mốc
        with torch.no_grad():                      # đánh giá, không cần gradient
            print(buoc, round(lf(m(Xtr).squeeze(1), ytr).item(), 4), round(dung(Xtr, ytr), 4), round(dung(Xte, yte), 4))   # loss, train, test
    opt.zero_grad()                                # 1. xóa gradient cũ
    loss = lf(m(Xtr).squeeze(1), ytr)              # 2. lượt xuôi và loss
    loss.backward()                                # 3. lượt ngược
    opt.step()                                     # 4. cập nhật

Output mong đợi:

```text
0 0.6924 0.5 0.3444
100 0.3181 0.8476 0.8222
1000 0.1711 0.9476 0.9556
3000 0.1567 0.9476 0.9444
```

<a name="trang-4" id="trang-4"></a>

## Trang 4 · DataLoader, nhiều lớp và GPU

DataLoader và 10 chữ số viết tay.

*Đọc kèm trên web: [Trang 4 · DataLoader, nhiều lớp và GPU](https://ai-and-robotics.vercel.app/ai-engineer/ch25-pytorch-can-ban/04-dataloader-va-gpu/)*

In [ ]:
from sklearn.datasets import load_digits                    # chữ số 8x8
from torch.utils.data import TensorDataset, DataLoader      # chia lô
dg = load_digits()                                          # 1797 ảnh
X = dg.data / 16.0                                          # đưa về 0 tới 1
A, B, C, E = train_test_split(X, dg.target, test_size=0.2, random_state=0, stratify=dg.target)   # 1437 train, 360 test
torch.manual_seed(0)                                        # seed
dl = DataLoader(TensorDataset(torch.tensor(A, dtype=torch.float32), torch.tensor(C)), batch_size=64, shuffle=True)   # lô 64
net = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))   # 10 logit
opt = torch.optim.Adam(net.parameters(), lr=1e-3)           # Adam
ce = nn.CrossEntropyLoss()                                  # softmax + cross-entropy
Bt, Et = torch.tensor(B, dtype=torch.float32), torch.tensor(E)   # test
print(len(dl), sum(p.numel() for p in net.parameters()))   # số lô, số tham số
for ep in range(1, 31):                                     # 30 epoch
    for xb, yb in dl:                                       # mỗi vòng một lô
        opt.zero_grad()                                     # xóa gradient
        loss = ce(net(xb), yb)                              # loss
        loss.backward()                                     # lượt ngược
        opt.step()                                          # cập nhật
    if ep in (1, 5, 10, 30):                                # in vài mốc
        with torch.no_grad():                               # đánh giá
            print(ep, round(loss.item(), 4), round((net(Bt).argmax(1) == Et).float().mean().item(), 4))   # loss lô cuối, test

Output mong đợi:

```text
23 4810
1 2.1383 0.5861
5 1.1455 0.8944
10 0.5501 0.9278
30 0.1534 0.9583
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Bật GPU trên Colab, chuyển net và từng lô sang cuda. Có nhanh hơn không?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Đổi batch_size thành 16. Có bao nhiêu lô mỗi epoch?</p>
</div>

In [ ]:
thiet_bi = "cuda" if torch.cuda.is_available() else "cpu"   # chọn thiết bị
print(thiet_bi)                                             # cpu hoặc cuda

Output mong đợi:

```text
cpu
```

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics.vercel.app/ai-engineer/ch25-pytorch-can-ban/05-bai-tap/)*

### Bài 25.1 · Autograd trên một đa thức

Với $y = x^3 + 2x$ tại $x = 2$, tính tay $dy/dx$ rồi kiểm bằng backward.
<details><summary>Xem đáp án</summary>

$dy/dx = 3x^2 + 2 = 14$, $y = 12$. Code ở ô ngay dưới.
</details>

In [ ]:
x = torch.tensor(2.0, requires_grad=True)   # biến
y = x ** 3 + 2 * x                          # đa thức
y.backward()                                # đạo hàm
print(y.item(), x.grad.item())              # 12 và 14

Output mong đợi:

```text
12.0 14.0
```

### Bài 25.2 · Thay SGD bằng Adam

Mạng trang 3, Adam lr 0,01, 1000 bước. In loss, train, test.
<details><summary>Xem đáp án</summary>

Loss 0,151, train 94,3%, test 94,4%: chỉ đổi một dòng. Code ở ô ngay dưới.
</details>

In [ ]:
def chay(tao_opt, xoa=True, n=1000):                     # huấn luyện lại từ đầu
    torch.manual_seed(0)                                 # cùng khởi tạo
    m = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))   # mạng trang 3
    opt = tao_opt(m.parameters())                        # optimizer tùy chọn
    for _ in range(n):                                   # n bước
        if xoa:                                          # có xóa gradient không
            opt.zero_grad()                              # xóa
        loss = lf(m(Xtr).squeeze(1), ytr)                # loss
        loss.backward()                                  # lượt ngược
        opt.step()                                       # cập nhật
    with torch.no_grad():                                # đánh giá
        d = lambda X, y: ((m(X).squeeze(1) > 0).float() == y).float().mean().item()   # độ chính xác
        print(round(lf(m(Xtr).squeeze(1), ytr).item(), 4), round(d(Xtr, ytr), 4), round(d(Xte, yte), 4))   # loss, train, test
chay(lambda p: torch.optim.Adam(p, lr=0.01))             # Adam

Output mong đợi:

```text
0.151 0.9429 0.9444
```

### Bài 25.3 · Quên zero_grad

Mạng trang 3, SGD lr 0,5, 1000 bước, bỏ zero_grad.
<details><summary>Xem đáp án</summary>

Loss kẹt 0,4852, train 70%, test 83,3%, không có lỗi nào được báo. Code ở ô ngay dưới.
</details>

In [ ]:
chay(lambda p: torch.optim.SGD(p, lr=0.5), xoa=False)    # không xóa gradient

Output mong đợi:

```text
0.4852 0.7 0.8333
```